# 이격도

In [ ]:
import datetime
import concurrent.futures
import time
import FinanceDataReader as fdr
import pandas as pd
from tqdm import tqdm


def get_target_stocks():
    """코스피 및 코스닥에 상장된 정상 종목 목록을 가져옵니다."""
    print("시장 데이터 로딩 중...")
    df_krx = fdr.StockListing("KRX")

    # ETN, ETF, 우선주 등을 제외하고 보통주 위주로 필터링
    filtered_df = df_krx[
        df_krx["Market"].isin(["KOSPI", "KOSDAQ"])
        & df_krx["Code"].str.isdigit()  # 숫자로만 된 코드 (보통주)
        & ~df_krx["Name"].str.contains("우|스팩|ETN|ETF")
    ]
    return filtered_df[["Code", "Name", "Market"]].to_dict("records")


def check_disparity_signal(stock, criteria_days=20, threshold=92.0):
    """개별 종목의 이격도를 계산하여 매수 신호 여부를 판별합니다."""
    symbol = stock["Code"]
    name = stock["Name"]
    market = stock["Market"]

    end_date = datetime.date.today().strftime("%Y-%m-%d")
    start_date = (datetime.date.today() - datetime.timedelta(days=100)).strftime(
        "%Y-%m-%d"
    )

    df = None
    retries = 3  # 최대 재시도 횟수

    # 네트워크 지연 및 차단 방지를 위한 재시도 루프
    for i in range(retries):
        try:
            # fdr 내부의 requests 타임아웃을 간접적으로 제어하거나 지연이 발생하면 익셉션 처리
            # 5초 이내에 응답이 없으면 에러를 발생시키도록 감싸는 형태가 안전합니다.
            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as executor:
                future = executor.submit(
                    fdr.DataReader, symbol, start=start_date, end=end_date
                )
                df = future.result(timeout=5)  # 5초 타임아웃 설정
            if df is not None and not df.empty:
                break
        except concurrent.futures.TimeoutError:
            # 타임아웃 발생 시 잠시 대기 후 재시도
            time.sleep(0.5 * (i + 1))
            continue
        except Exception:
            return None

    # 모든 재시도가 실패했거나 데이터가 없는 경우
    if df is None or df.empty or len(df) < criteria_days:
        return None

    try:
        # 20일 이동평균선 계산
        df["MA20"] = df["Close"].rolling(window=criteria_days).mean()

        latest_row = df.iloc[-1]
        close_price = latest_row["Close"]
        ma20_price = latest_row["MA20"]

        if pd.isna(ma20_price) or ma20_price == 0:
            return None

        # 이격도 계산
        disparity = (close_price / ma20_price) * 100

        if disparity <= threshold:
            return {
                "종목코드": symbol,
                "종목명": name,
                "시장": market,
                "현재가": int(close_price),
                "20일이평": round(ma20_price, 2),
                "이격도(%)": round(disparity, 2),
            }
    except Exception:
        return None

    return None


def main():
    stocks = get_target_stocks()
    print(f"총 {len(stocks)}개 종목 검사 시작...")

    buy_signals = []

    # 서버 차단을 방지하기 위해 max_workers를 10~15 내외로 조절하는 것이 안전합니다.
    with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
        futures = {
            executor.submit(check_disparity_signal, stock): stock
            for stock in stocks
        }

        for future in tqdm(
            concurrent.futures.as_completed(futures), total=len(futures)
        ):
            try:
                # 전체 루프가 대기 상태에 빠지는 것을 방지하기 위해 여기에도 타임아웃 처리
                result = future.result(timeout=7)
                if result:
                    buy_signals.append(result)
            except Exception:
                # 타임아웃이나 에러가 발생한 종목은 과감히 패스하고 다음 종목으로 진행
                continue

    if buy_signals:
        result_df = pd.DataFrame(buy_signals)
        result_df = result_df.sort_values(by="이격도(%)", ascending=True)

        print(f"\n▲ 매수 신호 발생 종목 (총 {len(result_df)}개)")
        print(
            result_df.to_string(
                index=False,
                columns=["시장", "종목코드", "종목명", "현재가", "이격도(%)"],
            )
        )
    else:
        print("\n현재 이격도 92% 이하인 매수 신호 종목이 없습니다.")


if __name__ == "__main__":
    main()